# PG Sidecars — full configuration walkthrough

**Persona:** Data Engineer

**Goal:** show what the *default* `items_postgresql_driver_config` hides and how to drive it explicitly. We:

1. configure the three core PG sidecars at **catalog scope** — `geometries` (with statistics), `attributes` (with `external_id` extraction + columnar attribute schema) and `item_metadata` — *before* the collection is materialised, because `sidecars` is `Immutable` post-materialisation;
2. configure `items_schema` covering every parameter (strict + materialise + per-field `FieldDefinition` capabilities + constraints);
3. configure `items_write_policy` with `external_id` matcher and `on_conflict=new_version`;
4. POST a feature that **matches** the schema → `201`;
5. POST a feature that is **refused** by `strict_unknown_fields` → `422`;
6. demonstrate the dedup contract — `on_conflict=refuse_return` plus the chain `[external_id, geometry_hash]`. Three duplicate-POST paths, **all guaranteed never to mutate the existing row**:
   - same `properties.code` → `external_id` matcher → existing geoid returned (`200`);
   - same geometry (WKB) → `geometry_hash` matcher → existing geoid returned (`200`);
   - same `geoid` → hub PK collision → existing geoid surfaced as `409 ConflictError` carrying the existing row, **never** as a silent overwrite.

## Why this notebook exists

The platform's default-fast policy persists **zero** `collection_configs` rows on `POST /collections`; sidecars are resolved lazily at `ensure_storage()` from extension defaults. That is operationally efficient but makes the `GET .../effective` payload look empty for anyone who hasn't read the resolver. This notebook shows the configurations explicitly so the resulting `effective` payload is non-trivial and round-trippable.

## Auth

No password grant — `geoid-fe` rejects ROPC by design. Configuring sidecars / schema / write-policy and creating the catalog all require a sysadmin bearer; export it as `DYNASTORE_ADMIN_TOKEN` (or `DYNASTORE_SYSADMIN_TOKEN`). Without a token the notebook prints a clear note, skips the privileged steps, and points the feature POSTs at an existing catalog/collection you supply via `CATALOG_ID` / `COLLECTION_ID` env vars (e.g. a review-env catalog where these configs are already in place).

In [ ]:
import os
import json
import time
import uuid

import httpx

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=False)
BASE_URL = os.environ.get(

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
    "DYNASTORE_BASE_URL",
    "http://localhost:8080",
).rstrip("/")
ADMIN_TOKEN = (
    os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or ""
)
HAVE_ADMIN = bool(ADMIN_TOKEN)

CATALOG_ID    = os.environ.get("CATALOG_ID")    or f"pg-sidecars-{uuid.uuid4().hex[:8]}"
COLLECTION_ID = os.environ.get("COLLECTION_ID") or "sensors"
RUN_ID        = uuid.uuid4().hex[:8]
EXTERNAL_CODE = f"sensor-{RUN_ID}"

headers = {"Content-Type": "application/json"}
if HAVE_ADMIN:
    headers["Authorization"] = f"Bearer {ADMIN_TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

def _show(r, label):
    body = r.text
    if len(body) > 400:
        body = body[:400] + "…"
    print(f"[{label}] {r.request.method} {r.request.url.path} → {r.status_code}")
    if body and r.status_code >= 400:
        print(f"    body: {body}")
    return r

print(f"BASE_URL    = {BASE_URL}")
print(f"HAVE_ADMIN  = {HAVE_ADMIN}")
print(f"CATALOG_ID  = {CATALOG_ID}")
print(f"COLLECTION  = {COLLECTION_ID}")
print(f"EXTERNAL_ID = {EXTERNAL_CODE}")

## 1. Bootstrap catalog

Create an ephemeral catalog. Wait for `provisioning_status=ready` so subsequent config PUTs see the IAM/seed pipeline finished.

Catalog creation requires sysadmin; if `HAVE_ADMIN=False` the cell short-circuits and the rest of the notebook becomes a *read-only walkthrough* against the published config surface.

In [ ]:
if not HAVE_ADMIN:
    print("NOTE: no admin token — skipping catalog creation. Subsequent config PUTs will be skipped too;\n"
          "      the feature POSTs at the end will hit the review env's existing public collections.")
else:
    r = client.post("/stac/catalogs", content=json.dumps({
        "id": CATALOG_ID,
        "type": "Catalog",
        "title": "PG sidecars walkthrough",
        "description": "Ephemeral catalog used by pg_sidecars_schema_versioning notebook.",
        "stac_version": "1.0.0",
    }))
    _show(r, "create catalog")
    assert r.status_code == 201, r.text

    for _ in range(30):
        rr = client.get(f"/stac/catalogs/{CATALOG_ID}")
        if rr.status_code == 200 and rr.json().get("provisioning_status") == "ready":
            print("catalog is ready")
            break
        time.sleep(1)
    else:
        print("WARN: catalog still provisioning after 30s — continuing anyway")

## 2. (Phase 3) Sidecars are DERIVED — not authored

`ItemsPostgresqlDriverConfig.sidecars` is a **Computed** (read-only) field. You no longer PUT a `sidecars` block — the PG driver derives the whole sidecar plan from `items_write_policy` + `items_schema` + the collection type at `ensure_storage()` time, and any caller-supplied `sidecars` on the config-write path is stripped.

How the old knobs map now:

| old authorable sidecar knob | Phase-3 source of truth |
|---|---|
| geometry spatial-cell index columns | DERIVED from `items_write_policy.derive.spatial_cells` (a geohash/h3/s2 cell + resolution). The stored geohash column width = the geohash cell's resolution. |
| geometry/attribute statistics | `items_write_policy.derive.geometry_stats` / `attribute_stats` |
| typed attribute columns | `items_schema.fields` (+ `materialize_fields_as_columns=True`) |
| identity / validity columns | `items_write_policy` identity / `validity` |

So steps 3 (`items_schema`) and 4 (`items_write_policy`) below are now the ONLY things you author; the resolved sidecars show up read-only under the PG driver node's `physical` projection in the composed view (step 6).

In [ ]:
# Phase 3: there is no driver-config PUT with a sidecars block anymore —
# sidecars are derived (Computed/read-only). Shape them via items_schema
# (step 3) and items_write_policy (step 4). The geohash storage column is
# driven by a geohash spatial cell on items_write_policy.derive.spatial_cells.
print('Phase 3: sidecars are derived from policy/schema — nothing to PUT here.')

## 3. PUT `items_schema` at **catalog scope** — covering every parameter

`ItemsSchema` carries:

- `level` — entity level (`item` for features);
- `fields` — `Dict[str, FieldDefinition]`. Each `FieldDefinition` exercises every property: `name`, `data_type`, `title`, `description`, `capabilities` (filterable / sortable / aggregatable / indexed / spatial), `required`, `unique`, `aggregations`, `expose`;
- `exclude_fields` — fields to remove from the public surface;
- `metadata_fields` — arbitrary extra metadata bag;
- `allow_app_level_enforcement=True` — opt in to service-layer required/unique checks (PG cannot block unknown JSON keys natively);
- `strict_unknown_fields=True` — refuse writes carrying undeclared properties (HTTP 422);
- `materialize_fields_as_columns=True` — lift every declared field into a native PG column on the attributes sidecar;
- `constraints` — declarative field constraints (left empty here; identity is bound via `items_write_policy.external_id_field` to keep the demo readable).

In [ ]:
ITEMS_SCHEMA = {
    "level": "item",
    "fields": {
        "code": {
            "name": "code",
            "alias": "sensor_code",
            "title": {"en": "Sensor code", "fr": "Code du capteur"},
            "description": "Stable external identifier carried by every reading.",
            "capabilities": ["filterable", "sortable", "indexed"],
            "data_type": "text",
            "expose": True,
            "required": True,
            # Uniqueness is owned by the attributes sidecar via `index_external_id` (partial on active rows).
            # Setting `unique=True` here would create a competing hard constraint that would 409
            # the v2 insert *before* the matcher dispatches `new_version`.
            "unique": False,
        },
        "temperature": {
            "name": "temperature",
            "title": "Temperature (°C)",
            "description": "Latest reading.",
            "capabilities": ["filterable", "sortable", "aggregatable"],
            "data_type": "float",
            "expose": True,
            "required": False,
            "aggregations": ["min", "max", "avg", "count"],
        },
        "status": {
            "name": "status",
            "title": "Operational status",
            "capabilities": ["filterable", "groupable"],
            "data_type": "text",
            "expose": True,
            "required": False,
        },
        "installed_on": {
            "name": "installed_on",
            "title": "Installation date",
            "capabilities": ["filterable", "sortable"],
            "data_type": "timestamp",
            "expose": True,
            "required": False,
        },
    },
    "exclude_fields": [],
    "metadata_fields": {"source": "notebook:pg_sidecars_schema_versioning"},
    "allow_app_level_enforcement": True,
    "strict_unknown_fields": True,
    "materialize_fields_as_columns": True,
    "constraints": [],
}

if HAVE_ADMIN:
    r = client.put(
        f"/configs/catalogs/{CATALOG_ID}/plugins/items_schema",
        json=ITEMS_SCHEMA,
    )
    _show(r, "PUT items_schema @ catalog")
    assert r.status_code in (200, 201), r.text
else:
    print("skipped: no admin token")

## 4. PUT `items_write_policy` at **catalog scope** — read-through dedup

One policy expresses three dedup rules:

```json
{
  "identity_matchers": ["external_id", "geometry_hash"],
  "external_id_field": "properties.code",
  "on_conflict": "refuse_return"
}
```

| Incoming payload | Resolved by | Result |
|---|---|---|
| Same `properties.code` as existing row | `external_id` matcher (attributes sidecar) | existing geoid returned, no write |
| Same WKB geometry as existing row     | `geometry_hash` matcher — uses the STORED GENERATED `geometry_hash CHAR(64)` column the geometries sidecar maintains automatically (SHA256 of `ST_AsBinary(geom)`) | existing geoid returned, no write |
| Same `geoid` as existing row          | hub primary-key short-circuit (geoid is unique by construction; it is *not* an `IdentityMatcher` enum value, so it does not flow through the chain) | `409 ConflictError` whose body cites the existing geoid — guaranteed never a silent overwrite |

Notes:

- **Chain order is "first match wins."** `external_id` is cheaper than `geometry_hash`, so list it first.
- **No extra sidecar wiring.** `geometry_hash` is auto-populated by the geometries sidecar configured in step 2; `external_id` extraction is wired via `attributes.external_id_field="properties.code"`.
- **Geoid collisions are always conflicts, never writes.** `refuse_return` covers semantic duplicates (matcher chain hits); a literal geoid collision is a *physical* primary-key collision on the hub table and surfaces as `409`. The combined invariant: **no incoming POST can ever overwrite an existing row's geoid-keyed identity.**
- **`refuse_return` is the idempotent-read-through policy** (see `WriteConflictPolicy` in `driver_config.py`): caller-visible result is the *existing* row's geoid in the body, no DB mutation.

> **PUT response shape.** Config-API PUTs return `200` with an empty body when accepted — the write is fire-and-forget. Inspect via GET `…/plugins/{plugin_id}` (catalog-scope payload) or `…/effective` (waterfall-resolved payload). Step 6 below does that.

In [ ]:
WRITE_POLICY = {
    "identity_matchers": ["external_id", "geometry_hash"],
    "external_id_field": "properties.code",
    "on_conflict": "refuse_return",
    # Geometry write-time behaviour lives on the policy, not on the sidecar
    # config. Defaults shown explicitly here for clarity.
    "geometries": {
        "invalid_geom_policy": "attempt_fix",
        "srid_mismatch_policy": "transform",
        "allowed_geometry_types": [],
        "simplification_algorithm": None,
        "simplification_tolerance": None,
        "remove_redundant_vertices": False,
    },
}

if HAVE_ADMIN:
    r = client.put(
        f"/configs/catalogs/{CATALOG_ID}/plugins/items_write_policy",
        json=WRITE_POLICY,
    )
    _show(r, "PUT items_write_policy @ catalog")
    assert r.status_code in (200, 201), r.text
else:
    print("skipped: no admin token")

## 5. Create the collection — materialisation derives + snapshots the sidecar plan

POST a default collection. The PG driver's `ensure_storage()` runs here, DERIVES the sidecar plan from `items_schema` + `items_write_policy`, and stamps the resolved (Computed) `sidecars` list onto the collection. You never authored it; you can only read it back.

In [ ]:
if HAVE_ADMIN:
    r = client.post(
        f"/stac/catalogs/{CATALOG_ID}/collections",
        content=json.dumps({
            "id": COLLECTION_ID,
            "type": "Collection",
            "stac_version": "1.0.0",
            "title": "Sensors — pg sidecars walkthrough",
            "description": "Versioned sensors with explicit PG sidecar configuration.",
            "license": "proprietary",
            "extent": {
                "spatial":  {"bbox":     [[-180, -90, 180, 90]]},
                "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
            },
        }),
    )
    _show(r, "create collection")
    assert r.status_code == 201, r.text
else:
    print("skipped: no admin token")

## 6. Verify effective configs — the resolved view shows the DERIVED sidecars

The PG driver node carries a read-only `physical` projection showing the resolved sidecars + their derived fields (geohash precision, identity columns, statistics) — derived from the policy/schema, not authored. This is how an operator SEES the physical realization without being able to author it.

In [ ]:
for plugin_id in (
    "items_postgresql_driver_config",
    "items_schema",
    "items_write_policy",
):
    r = client.get(
        f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/{plugin_id}/effective"
    )
    print(f"\n=== {plugin_id} ===")
    if r.status_code != 200:
        print(f"  {r.status_code}: {r.text[:200]}")
        continue
    payload = r.json()
    resolved = payload.get("resolved", payload)
    print(json.dumps(resolved, indent=2)[:1500])

## 7. Feature 1 — payload **matches** the schema → `201`

All declared properties are present (`code`, `temperature`, `status`, `installed_on`), no rogue field, geometry is a valid Point. The attributes sidecar extracts `properties.code` into the `external_id` column.

In [ ]:
ITEMS_URL = f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items"

feature_v1 = {
    "type": "Feature",
    "id": EXTERNAL_CODE,
    "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},
    "bbox": [12.49, 41.89, 12.49, 41.89],
    "properties": {
        "code":         EXTERNAL_CODE,
        "temperature":  22.3,
        "status":       "nominal",
        "installed_on": "2024-04-01T00:00:00Z",
    },
}

r = client.post(ITEMS_URL, content=json.dumps(feature_v1))
_show(r, "feature v1")
assert r.status_code in (200, 201), r.text

## 8. Feature 2 — payload carries an undeclared property → `422`

`unauthorized_field` is not in `items_schema.fields`. With `strict_unknown_fields=True` and `allow_app_level_enforcement=True` the service layer (`item_service.upsert`) refuses the write and returns RFC 9457 Problem Details listing the offending field. No partial DB state is created.

In [ ]:
rogue_feature = {
    "type": "Feature",
    "id": f"{EXTERNAL_CODE}-rogue",
    "geometry": {"type": "Point", "coordinates": [12.50, 41.90]},
    "properties": {
        "code":               f"{EXTERNAL_CODE}-rogue",
        "temperature":        99.9,
        "status":             "nominal",
        "installed_on":       "2024-04-01T00:00:00Z",
        "unauthorized_field": "BOOM",
    },
}

r = client.post(ITEMS_URL, content=json.dumps(rogue_feature))
_show(r, "feature rogue")
assert r.status_code == 422, f"expected 422 strict-mode rejection, got {r.status_code}: {r.text[:400]}"
problem = r.json()
print("\nProblem Details:")
print(json.dumps(problem, indent=2)[:600])

## 9. Three dedup paths — each returns the existing geoid, no DB mutation

Capture the geoid of the row inserted in step 7, then exercise each matcher in the chain. Every duplicate POST should return `200` with the **same** geoid.


In [ ]:
# Read back the v1 row to capture its server-assigned geoid
r = client.get(ITEMS_URL, params={"filter": f"code='{EXTERNAL_CODE}'"})
assert r.status_code == 200, r.text
v1_rows = r.json().get("features", [])
assert len(v1_rows) == 1, f"expected exactly one row after step 7, got {len(v1_rows)}"
EXISTING_GEOID = v1_rows[0].get("id")
print(f"existing geoid = {EXISTING_GEOID}")

# --- Case A: same external_id, different payload --------------------------
case_a = {
    "type": "Feature",
    "id": EXTERNAL_CODE,
    "geometry": {"type": "Point", "coordinates": [13.00, 42.00]},  # different geom
    "bbox": [13.00, 42.00, 13.00, 42.00],
    "properties": {
        "code":         EXTERNAL_CODE,                              # same external_id
        "temperature":  99.9,
        "status":       "nominal",
        "installed_on": "2024-04-01T00:00:00Z",
    },
}
r = client.post(ITEMS_URL, content=json.dumps(case_a))
_show(r, "case A — same external_id")
assert r.status_code in (200, 201), r.text
returned = r.json()
print(f"  returned geoid = {returned.get('id')}")
assert returned.get("id") == EXISTING_GEOID, "external_id matcher must return the existing geoid"

# --- Case B: different external_id, same geometry as v1 -------------------
case_b = {
    "type": "Feature",
    "id": f"{EXTERNAL_CODE}-other",
    "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},   # same geom as v1
    "bbox": [12.49, 41.89, 12.49, 41.89],
    "properties": {
        "code":         f"{EXTERNAL_CODE}-other",                   # different external_id
        "temperature":  10.0,
        "status":       "nominal",
        "installed_on": "2024-04-01T00:00:00Z",
    },
}
r = client.post(ITEMS_URL, content=json.dumps(case_b))
_show(r, "case B — same geometry")
assert r.status_code in (200, 201), r.text
returned = r.json()
print(f"  returned geoid = {returned.get('id')}")
assert returned.get("id") == EXISTING_GEOID, "geometry_hash matcher must return the existing geoid"

# --- Case C: explicit geoid collision -------------------------------------
# Clients normally do not post a geoid; the platform generates it. When one is
# supplied that matches an existing row, refuse_return echoes the row back.
case_c = {
    "type": "Feature",
    "id": EXISTING_GEOID,                                            # collide on geoid
    "geometry": {"type": "Point", "coordinates": [14.00, 43.00]},
    "bbox": [14.00, 43.00, 14.00, 43.00],
    "properties": {
        "code":         f"{EXTERNAL_CODE}-geoid-collide",
        "temperature":  -5.0,
        "status":       "nominal",
        "installed_on": "2024-04-01T00:00:00Z",
    },
}
r = client.post(ITEMS_URL, content=json.dumps(case_c))
_show(r, "case C — same geoid")
# Geoid collision is a guaranteed-refuse use case. Two surface forms are both valid
# proofs that the existing row was not overwritten:
#   * 200/201 — refuse_return echoed the existing row (matcher-chain path)
#   * 409    — hub PK collision raised ConflictError citing the existing geoid
assert r.status_code in (200, 201, 409), r.text
if r.status_code in (200, 201):
    returned = r.json()
    print(f"  returned geoid = {returned.get('id')}")
    assert returned.get("id") == EXISTING_GEOID, "case C must return the existing geoid"
else:
    body = r.text
    print(f"  conflict body = {body[:300]}")
    assert EXISTING_GEOID in body, (
        f"case C 409 body must reference the existing geoid {EXISTING_GEOID!r}, got {body[:300]!r}"
    )

## 10. Verify there is still exactly one row, untouched

Three duplicate POSTs in step 9 must not have produced any new rows or mutated the original. Filter on `code` and assert: **one** active row, geoid = `EXISTING_GEOID`, geometry/temperature/status = the values from step 7.

In [ ]:
r = client.get(ITEMS_URL, params={"filter": f"code='{EXTERNAL_CODE}'"})
assert r.status_code == 200, r.text
rows = r.json().get("features", [])
print(f"rows for code={EXTERNAL_CODE}: {len(rows)} (expected 1)")
assert len(rows) == 1, f"expected exactly one active row, got {len(rows)}"

f = rows[0]
coords = f.get("geometry", {}).get("coordinates")
props  = f.get("properties", {})
print(f"  geoid={f.get('id')} coords={coords} temperature={props.get('temperature')}")

assert f.get("id")               == EXISTING_GEOID,           "geoid must be unchanged"
assert coords                    == [12.49, 41.89],           f"geometry must be untouched, got {coords}"
assert props.get("temperature")  == 22.3,                     f"temperature must be untouched, got {props.get('temperature')}"
assert props.get("status")       == "nominal",                f"status must be untouched, got {props.get('status')}"

print("refuse_return confirmed: three duplicate POSTs, zero mutations.")

## 11. Teardown

Drop the catalog (cascades to collection + every config we PUT). Skipped when the notebook ran read-only.

In [ ]:
if HAVE_ADMIN:
    r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
    _show(r, "DELETE catalog")
    assert r.status_code in (204, 404), r.text
else:
    print("skipped: no admin token")

client.close()
print("done.")